# 08 - Normalizing Flows

Mean-field VI is simple, but it can be too restrictive.
**Normalizing flows** make a simple base distribution more expressive by applying invertible transformations.

This notebook covers:
1. Why expressive variational families are useful
2. Change-of-variables intuition
3. How a flow transforms density
4. A simple 1D example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: The basic idea

Start with a simple variable
$$u im p_0(u)$$
such as a standard Gaussian.

Then transform it with an invertible map
$$z = f(u).$$

The distribution of $z$ can become much more flexible than the original base distribution.

## Part 2: Change of variables

If $z = f(u)$ and $f$ is invertible, then
$$
q(z) = p_0(u) eft| et rac{artial f^{-1}(z)}{artial z} 
ight|.
$$

In 1D, this becomes
$$
q(z) = p_0(u) eft| rac{du}{dz} 
ight|.
$$

The Jacobian term tells us how density stretches or compresses under the transformation.

In [ ]:
u = np.linspace(-3, 3, 1000)
base_pdf = stats.norm.pdf(u, 0, 1)

def flow(u):
    return u + 0.8 * np.tanh(2 * u)

z = flow(u)

plt.figure(figsize=(9, 4))
plt.plot(u, z, color='purple', linewidth=2)
plt.axline((0, 0), slope=1, color='gray', linestyle='--')
plt.title('A simple invertible 1D flow: z = u + 0.8 tanh(2u)')
plt.xlabel('u')
plt.ylabel('z')
plt.show()

In [ ]:
# Numerical density transformation by sampling
u_samples = np.random.normal(size=20000)
z_samples = flow(u_samples)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(u_samples, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].plot(u, base_pdf, color='navy', linewidth=2)
axes[0].set_title('Base distribution p0(u)')
axes[0].set_xlabel('u')

axes[1].hist(z_samples, bins=60, density=True, alpha=0.7, color='darkorange', edgecolor='black')
axes[1].set_title('Transformed distribution q(z)')
axes[1].set_xlabel('z')

plt.tight_layout()
plt.show()

## Part 3: Why this helps in VI

Instead of using a simple Gaussian variational family, we can define
$$
u im p_0(u), quad z = f_K irc dots irc f_1(u).
$$

By stacking transformations, the approximation becomes much more expressive.
This helps capture skewness, heavy tails, and more complex structure.

In [ ]:
def flow2(u):
    return flow(u) + 0.25 * np.sin(3 * flow(u))

z2_samples = flow2(u_samples)

plt.figure(figsize=(8, 4))
plt.hist(z_samples, bins=60, density=True, alpha=0.6, color='darkorange', edgecolor='black', label='one flow step')
plt.hist(z2_samples, bins=60, density=True, alpha=0.5, color='seagreen', edgecolor='black', label='two flow-like steps')
plt.title('More transformations => more expressive distributions')
plt.xlabel('z')
plt.ylabel('density')
plt.legend()
plt.show()

## Part 4: Trade-offs

Normalizing flows improve flexibility, but they also require:
- invertible transformations
- tractable Jacobian determinants
- more computation than mean-field VI

So they trade simplicity for expressiveness.

## Summary

What to remember:
1. Normalizing flows start with a simple base distribution
2. Invertible transformations make the distribution more expressive
3. The Jacobian term adjusts density correctly
4. Flows are a powerful upgrade over simple mean-field approximations

In [ ]:
# Exercises
# 1) Change the flow function and inspect the resulting histogram.
# 2) Stack more transformations and see how the shape changes.
# 3) Think about why invertibility is important for density evaluation.

pass